In [1]:
import re
from pathlib import Path
 
import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.io import loadmat

OMP: Warning #80: OMP_NUM_THREADS="0": value too small.
OMP: Info #104: OMP_NUM_THREADS value "1" will be used.


In [2]:
EVENT_DIR = Path('/root/autodl-tmp/Test_on_Event/chichi/Chichi')
 
GT_FILES = {
    'CHI'  : EVENT_DIR / 's1999CHICHI01CHIx.mat',
    'HAYE' : EVENT_DIR / 's1999CHICHI01HAYE.mat',
    'JOHN' : EVENT_DIR / 's1999CHICHI01JOHN.mat',
    'MA'   : EVENT_DIR / 's1999CHICHI01MAxx.mat',
    'SEKI' : EVENT_DIR / 's1999CHICHI01SEKI.mat',
    'WU'   : EVENT_DIR / 's1999CHICHI01WUxx.mat',
    'ZENG' : EVENT_DIR / 's1999CHICHI01ZENG.mat',
    'MA02' : EVENT_DIR / 's1999CHICHI02MAxx.mat',
}
 
GRID_SIZE        = 150
RESOLUTION       = 0.01
SUBFAULT_SIZE_KM = 2.0
SLIP_RATIO       = 0.10
MIN_PIX_ANGLE    = 20
REP_FRACTIONS    = [0.25, 0.50, 0.75, 1.0]   # representative time fractions


In [3]:
def _deep_flatten(v):
    if v is None: return np.array([], dtype=np.float64)
    try: a = np.asarray(v)
    except: return np.array([], dtype=np.float64)
    if a.size == 0: return np.array([], dtype=np.float64)
    if a.dtype == object:
        parts = [_deep_flatten(c) for c in a.flat if c is not None]
        return np.concatenate(parts) if parts else np.array([], dtype=np.float64)
    try: return a.astype(np.float64).flatten()
    except: return np.array([], dtype=np.float64)
 
def _safe_scalar(v, default=0.0):
    a = _deep_flatten(v); return float(a[0]) if a.size > 0 else default
 
def _safe_pair(v, default=(0.0, 0.0)):
    a = _deep_flatten(v)
    return (float(a[0]), float(a[1])) if a.size >= 2 else default
 
def _seg_field_index(s, fields, aliases):
    for alias in aliases:
        pat = re.compile(r'^seg(\d+)' + re.escape(alias) + r'$')
        items = sorted([(int(m.group(1)), f) for f in fields
                        if (m := pat.match(f))], key=lambda x: x[0])
        if items: return items, alias
    return [], None
 
def _gather_seg_arrays(s, items):
    return [_deep_flatten(getattr(s, name)) for _, name in items]
 
def parse_srcmod_mat(mat_path):
    data = loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)
    top  = [k for k in data if not k.startswith('__')]
    if not top: return pd.DataFrame(), {}
    s = data[top[0]]; fields = getattr(s, '_fieldnames', [])
    fld = lambda n: getattr(s, n) if n in fields else None
 
    info = {
        'hypo_lat': _safe_scalar(fld('evLAT')),
        'hypo_lon': _safe_scalar(fld('evLON')),
        'hypo_dep_km': _safe_scalar(fld('evDPT'), 10.0),
        'strike': _safe_scalar(fld('srcAStke')),
        'dip': _safe_scalar(fld('srcDipAn'), 90.0),
        'rake': _safe_scalar(fld('srcARake')),
    }
    sd_g, ss_g = _safe_pair(fld('invDzDx'), (SUBFAULT_SIZE_KM, SUBFAULT_SIZE_KM))
    sd_g = SUBFAULT_SIZE_KM if sd_g <= 0 or sd_g > 50 else sd_g
    ss_g = SUBFAULT_SIZE_KM if ss_g <= 0 or ss_g > 50 else ss_g
    info['size_dip_km'] = sd_g; info['size_strike_km'] = ss_g
 
    if 'slipSPL' in fields and 'geoLAT' in fields:
        slip = _deep_flatten(fld('slipSPL'))
        lat  = _deep_flatten(fld('geoLAT'))
        lon  = _deep_flatten(fld('geoLON'))
        tinit = _deep_flatten(fld('tinitSPL'))
        strike_arr = np.full(len(slip), info['strike'])
        sd_arr = np.full(len(slip), sd_g); ss_arr = np.full(len(slip), ss_g)
    else:
        slip_i, _ = _seg_field_index(s, fields, ['SLIP'])
        lat_i,  _ = _seg_field_index(s, fields, ['geoLAT','LAT','Lat'])
        lon_i,  _ = _seg_field_index(s, fields, ['geoLON','LON','Lon'])
        stk_i,  _ = _seg_field_index(s, fields, ['AStke','STK','STRIKE'])
        dim_i,  _ = _seg_field_index(s, fields, ['DimWL','invDzDx'])
        tin_i,  _ = _seg_field_index(s, fields, ['TRUP','TINIT','Trup'])
        if not slip_i or not lat_i or not lon_i:
            print('  [parse] FAILED'); return pd.DataFrame(), info
 
        slip_a = _gather_seg_arrays(s, slip_i)
        lat_a  = _gather_seg_arrays(s, lat_i)
        lon_a  = _gather_seg_arrays(s, lon_i)
        stk_a  = _gather_seg_arrays(s, stk_i)  if stk_i  else [np.array([])]*len(slip_a)
        dim_a  = _gather_seg_arrays(s, dim_i)  if dim_i  else [np.array([])]*len(slip_a)
        tin_a  = _gather_seg_arrays(s, tin_i)  if tin_i  else [np.array([])]*len(slip_a)
 
        all_s,all_la,all_lo,all_stk,all_sd,all_ss,all_t=[],[],[],[],[],[],[]
        for k in range(len(slip_a)):
            sa=slip_a[k]; la=lat_a[k] if k<len(lat_a) else np.array([])
            lo=lon_a[k]  if k<len(lon_a) else np.array([])
            n=min(sa.size,la.size,lo.size)
            if n==0: continue
            def bc(a,fb):
                a=np.asarray(a,dtype=np.float64).flatten()
                if a.size>=n: return a[:n]
                if a.size==1: return np.full(n,float(a[0]))
                if a.size==0: return np.full(n,float(fb))
                return np.concatenate([a,np.full(n-a.size,float(a[-1]))])
            all_s.append(sa[:n]); all_la.append(la[:n]); all_lo.append(lo[:n])
            all_stk.append(bc(stk_a[k] if k<len(stk_a) else np.array([]),info['strike']))
            di=np.asarray(dim_a[k] if k<len(dim_a) else np.array([]),dtype=np.float64).flatten()
            sd_s,ss_s=(float(di[0]),float(di[1])) if di.size>=2 else (sd_g,ss_g)
            all_sd.append(np.full(n,sd_s)); all_ss.append(np.full(n,ss_s))
            all_t.append(bc(tin_a[k] if k<len(tin_a) else np.array([]),np.inf))
 
        if not all_s: return pd.DataFrame(), info
        slip=np.concatenate(all_s); lat=np.concatenate(all_la); lon=np.concatenate(all_lo)
        strike_arr=np.concatenate(all_stk); sd_arr=np.concatenate(all_sd)
        ss_arr=np.concatenate(all_ss); tinit=np.concatenate(all_t)
 
    if slip.size>0 and np.nanmean(slip)>10: slip=slip/100.0
    n=min(len(slip),len(lat),len(lon))
    rows=[{'lat':float(lat[k]),'lon':float(lon[k]),
           'size_dip_km':float(sd_arr[k]) if k<len(sd_arr) else sd_g,
           'size_strike_km':float(ss_arr[k]) if k<len(ss_arr) else ss_g,
           'strike':float(strike_arr[k]) if k<len(strike_arr) else info['strike'],
           'slip_m':float(slip[k]),
           'ttrg_s':float(tinit[k]) if k<len(tinit) and np.isfinite(tinit[k]) else np.inf}
          for k in range(n)]
    return pd.DataFrame(rows), info
 
 
def subfault_to_mask(df, eq_lat, eq_lon):
    half=GRID_SIZE*RESOLUTION*0.5
    lon_min,lon_max=eq_lon-half,eq_lon+half
    lat_min,lat_max=eq_lat-half,eq_lat+half
    mask=np.zeros((GRID_SIZE,GRID_SIZE),dtype=np.float32)
    if df.empty: return mask.astype(np.uint8)
    sm=float(df['slip_m'].max())
    if sm<=0: return mask.astype(np.uint8)
    dfa=df[df['slip_m']>=sm*SLIP_RATIO]
    dpx_lon=(lon_max-lon_min)/(GRID_SIZE-1); dpx_lat=(lat_max-lat_min)/(GRID_SIZE-1)
    m_lat=111.32; m_lon=111.32*np.cos(np.radians(eq_lat))
    for _,row in dfa.iterrows():
        cf=(row['lon']-lon_min)/(lon_max-lon_min)*(GRID_SIZE-1)
        rf=(row['lat']-lat_min)/(lat_max-lat_min)*(GRID_SIZE-1)
        rx=(row.get('size_strike_km',SUBFAULT_SIZE_KM)/m_lon)/dpx_lon*0.5
        ry=(row.get('size_dip_km',   SUBFAULT_SIZE_KM)/m_lat)/dpx_lat*0.5
        c0=max(0,int(np.floor(cf-rx))); c1=min(GRID_SIZE-1,int(np.ceil(cf+rx)))
        r0=max(0,int(np.floor(rf-ry))); r1=min(GRID_SIZE-1,int(np.ceil(rf+ry)))
        mask[r0:r1+1,c0:c1+1]=np.maximum(mask[r0:r1+1,c0:c1+1],row['slip_m']/sm)
    return (mask>0).astype(np.uint8)
 
 
def build_tv_gt(df, t_inf, t_first, eq_lat, eq_lon):
    """Time-varying GT at inference time t_inf."""
    if df.empty: return np.zeros((GRID_SIZE,GRID_SIZE),dtype=np.uint8),0
    sm=float(df['slip_m'].max())
    if sm<=0: return np.zeros((GRID_SIZE,GRID_SIZE),dtype=np.uint8),0
    t_orig=float(t_inf)+float(t_first)
    dfa=df[(df['ttrg_s']<=t_orig)&(df['slip_m']>=sm*SLIP_RATIO)]
    if dfa.empty: return np.zeros((GRID_SIZE,GRID_SIZE),dtype=np.uint8),0
    half=GRID_SIZE*RESOLUTION*0.5
    lon_min,lon_max=eq_lon-half,eq_lon+half
    lat_min,lat_max=eq_lat-half,eq_lat+half
    mask=np.zeros((GRID_SIZE,GRID_SIZE),dtype=np.float32)
    dpx_lon=(lon_max-lon_min)/(GRID_SIZE-1); dpx_lat=(lat_max-lat_min)/(GRID_SIZE-1)
    m_lat=111.32; m_lon=111.32*np.cos(np.radians(eq_lat))
    for _,row in dfa.iterrows():
        cf=(row['lon']-lon_min)/(lon_max-lon_min)*(GRID_SIZE-1)
        rf=(row['lat']-lat_min)/(lat_max-lat_min)*(GRID_SIZE-1)
        rx=(row.get('size_strike_km',SUBFAULT_SIZE_KM)/m_lon)/dpx_lon*0.5
        ry=(row.get('size_dip_km',   SUBFAULT_SIZE_KM)/m_lat)/dpx_lat*0.5
        c0=max(0,int(np.floor(cf-rx))); c1=min(GRID_SIZE-1,int(np.ceil(cf+rx)))
        r0=max(0,int(np.floor(rf-ry))); r1=min(GRID_SIZE-1,int(np.ceil(rf+ry)))
        mask[r0:r1+1,c0:c1+1]=np.maximum(mask[r0:r1+1,c0:c1+1],row['slip_m']/sm)
    return (mask>0).astype(np.uint8), len(dfa)
 
 
def compute_metrics(pred, gt):
    p,g=pred.astype(bool).ravel(),gt.astype(bool).ravel()
    tp=int((p&g).sum()); fp=int((p&~g).sum()); fn=int((~p&g).sum())
    eps=1e-8
    return {'IoU':tp/(tp+fp+fn+eps),'F1':2*tp/(2*tp+fp+fn+eps),
            'Precision':tp/(tp+fp+eps),'Recall':tp/(tp+fn+eps)}
 
 
def major_axis_deg(binary_mask):
    m=(np.asarray(binary_mask)>0).astype(np.uint8)
    if m.sum()<MIN_PIX_ANGLE: return None
    rows,cols=np.where(m)
    if len(rows)<5: return None
    pts=np.column_stack([cols,rows]).astype(np.float32)
    try: _,_,angle=cv2.fitEllipse(pts)
    except cv2.error: return None
    return float(angle%180.0)
 
 
def draw_ellipse_on_ax(ax, binary_mask, color='yellow', lw=1.5):
    m=(np.asarray(binary_mask)>0).astype(np.uint8)
    if m.sum()<MIN_PIX_ANGLE: return
    rows,cols=np.where(m)
    if len(rows)<5: return
    pts=np.column_stack([cols,rows]).astype(np.float32)
    try: (cx,cy),(MA,ma),angle=cv2.fitEllipse(pts)
    except cv2.error: return
    from matplotlib.patches import Ellipse as MplEllipse
    ell=MplEllipse(xy=(cx,cy),width=MA,height=ma,angle=angle,
                   edgecolor=color,facecolor='none',linewidth=lw)
    ax.add_patch(ell)

In [4]:
def plot_iou_curve(label, t_axis, pred_binary, mask_gt, eff_end, out_dir):
    T=len(t_axis)
    static_iou=np.array([compute_metrics(pred_binary[t],mask_gt)['IoU'] for t in range(T)])
 
    fig,ax=plt.subplots(figsize=(10,4))
    ax.plot(t_axis,static_iou,color='#1f77b4',lw=1.8,label='IoU vs static GT')
    ax.axvline(eff_end,color='#d62728',lw=1.2,ls=':',
               label=f'effective end = {eff_end:.1f} s')
    ax.axvspan(eff_end,float(t_axis.max()),alpha=0.10,color='#d62728')
    ax.set_xlabel('Time since first trigger (s)',fontsize=11)
    ax.set_ylabel('IoU',fontsize=11)
    ax.set_ylim(-0.02,1.02); ax.grid(alpha=0.3)
    ax.set_title(f'Ridgecrest — {label} — IoU curve',fontsize=12)
    ax.legend(fontsize=9)
    plt.tight_layout()
    out=out_dir/f'Ridgecrest_{label}_iou_curve.png'
    plt.savefig(out,dpi=150,bbox_inches='tight'); plt.close()
    print(f'  Saved: {out}')
 
 
def plot_static_compare(label, pred_final, mask_gt, out_dir):
    fig,axes=plt.subplots(1,3,figsize=(12,4))
 
    axes[0].imshow(pred_final,cmap='gray',origin='lower',vmin=0,vmax=1)
    axes[0].set_title('Pred (last valid frame)',fontsize=11)
    draw_ellipse_on_ax(axes[0],pred_final,'yellow')
 
    axes[1].imshow(mask_gt,cmap='gray',origin='lower',vmin=0,vmax=1)
    axes[1].set_title(f'GT static ({label})',fontsize=11)
    draw_ellipse_on_ax(axes[1],mask_gt,'cyan')
 
    rgb=np.zeros((*pred_final.shape,3),dtype=np.uint8)
    p=pred_final.astype(bool); g=mask_gt.astype(bool)
    rgb[p&g]=[80,200,80]    # TP green
    rgb[p&~g]=[220,60,60]   # FP red
    rgb[~p&g]=[60,120,220]  # FN blue
    axes[2].imshow(rgb,origin='lower')
    axes[2].set_title('TP=green  FP=red  FN=blue',fontsize=11)
 
    patches=[mpatches.Patch(color='#50c850',label='TP'),
             mpatches.Patch(color='#dc3c3c',label='FP'),
             mpatches.Patch(color='#3c78dc',label='FN')]
    axes[2].legend(handles=patches,loc='lower right',fontsize=8)
 
    for ax in axes: ax.axis('off')
    fig.suptitle(f'Ridgecrest — {label} — static comparison',fontsize=12)
    plt.tight_layout()
    out=out_dir/f'Ridgecrest_{label}_static_compare.png'
    plt.savefig(out,dpi=150,bbox_inches='tight'); plt.close()
    print(f'  Saved: {out}')
 
 
def plot_tv_compare(label, t_axis, pred_binary, df_gt,
                    eq_lat, eq_lon, t_first, eff_end, out_dir):
    # pick representative frames within valid window
    valid_idx=np.where(t_axis<=eff_end)[0]
    if len(valid_idx)==0: valid_idx=np.arange(len(t_axis))
    rep_idx=[valid_idx[min(int(f*len(valid_idx)),len(valid_idx)-1)]
             for f in REP_FRACTIONS]
 
    has_ttrg=(df_gt['ttrg_s']<np.inf).sum()>0
 
    n=len(rep_idx)
    fig,axes=plt.subplots(2,n,figsize=(3.2*n,6.5),squeeze=False)
 
    for col,ti in enumerate(rep_idx):
        t_val=float(t_axis[ti])
 
        # prediction
        axes[0,col].imshow(pred_binary[ti],cmap='gray',origin='lower',vmin=0,vmax=1)
        axes[0,col].set_title(f't={t_val:.1f}s',fontsize=9)
        axes[0,col].axis('off')
        draw_ellipse_on_ax(axes[0,col],pred_binary[ti],'yellow')
 
        # GT
        if has_ttrg:
            gt_t,n_act=build_tv_gt(df_gt,t_val,t_first,eq_lat,eq_lon)
            axes[1,col].imshow(gt_t,cmap='gray',origin='lower',vmin=0,vmax=1)
            axes[1,col].set_title(f'GT t={t_val:.1f}s',fontsize=9)
        else:
            gt_static=subfault_to_mask(df_gt,eq_lat,eq_lon)
            axes[1,col].imshow(gt_static,cmap='gray',origin='lower',vmin=0,vmax=1)
            axes[1,col].set_title(f'GT static ({label})',fontsize=9)
        axes[1,col].axis('off')
 
    axes[0,0].set_ylabel('Pred',fontsize=10)
    axes[1,0].set_ylabel('GT',  fontsize=10)
    fig.suptitle(f'Ridgecrest — {label} — time-varying comparison',fontsize=12)
    plt.tight_layout()
    out=out_dir/f'Ridgecrest_{label}_tv_compare.png'
    plt.savefig(out,dpi=150,bbox_inches='tight'); plt.close()
    print(f'  Saved: {out}')

In [5]:
def main():
    out_dir=EVENT_DIR/'figures'; out_dir.mkdir(exist_ok=True)
 
    pred_binary=np.load(EVENT_DIR/'pred_binary.npy')
    t_axis     =np.load(EVENT_DIR/'times_mid_valid.npy')
    meta       =np.load(EVENT_DIR/'event_metadata.npz')
    eq_lat     =float(meta['eq_lat']); eq_lon=float(meta['eq_lon'])
    t_first    =float(meta['t_first_trigger_s'])
    eff_end_abs=float(meta['effective_end_abs_s'])
    eff_end_inf=max(0.0,eff_end_abs-t_first)
 
    valid=t_axis<=eff_end_inf
    if not valid.any(): valid=np.ones(len(t_axis),dtype=bool)
    last_vi=int(np.where(valid)[0][-1])
    pred_final=pred_binary[last_vi]
 
    print(f'pred shape={pred_binary.shape}  '
          f'eff_end_inf={eff_end_inf:.1f}s  '
          f'last valid frame: t={t_axis[last_vi]:.1f}s')
    print(f'Output dir: {out_dir}\n')
 
    for label,mat_path in GT_FILES.items():
        print(f'── {label} ──')
        if not mat_path.exists():
            print('  [SKIP] file not found'); continue
        df_gt,info_gt=parse_srcmod_mat(mat_path)
        if df_gt.empty:
            print('  [SKIP] parse failed'); continue
 
        mask_gt=subfault_to_mask(df_gt,eq_lat,eq_lon)
        m=compute_metrics(pred_final,mask_gt)
        print(f'  Static IoU={m["IoU"]:.3f}  F1={m["F1"]:.3f}  '
              f'P={m["Precision"]:.3f}  R={m["Recall"]:.3f}')
 
        plot_iou_curve(label,t_axis,pred_binary,mask_gt,eff_end_inf,out_dir)
        plot_static_compare(label,pred_final,mask_gt,out_dir)
        plot_tv_compare(label,t_axis,pred_binary,df_gt,
                        eq_lat,eq_lon,t_first,eff_end_inf,out_dir)
        print()
 
    print(f'\nAll figures saved to {out_dir}')

In [6]:
main()

pred shape=(54, 150, 150)  eff_end_inf=37.2s  last valid frame: t=36.6s
Output dir: /root/autodl-tmp/Test_on_Event/chichi/Chichi/figures

── CHI ──
  Static IoU=0.587  F1=0.740  P=0.762  R=0.720
  Saved: /root/autodl-tmp/Test_on_Event/chichi/Chichi/figures/Ridgecrest_CHI_iou_curve.png
  Saved: /root/autodl-tmp/Test_on_Event/chichi/Chichi/figures/Ridgecrest_CHI_static_compare.png
  Saved: /root/autodl-tmp/Test_on_Event/chichi/Chichi/figures/Ridgecrest_CHI_tv_compare.png

── HAYE ──
  Static IoU=0.394  F1=0.565  P=0.823  R=0.430
  Saved: /root/autodl-tmp/Test_on_Event/chichi/Chichi/figures/Ridgecrest_HAYE_iou_curve.png
  Saved: /root/autodl-tmp/Test_on_Event/chichi/Chichi/figures/Ridgecrest_HAYE_static_compare.png
  Saved: /root/autodl-tmp/Test_on_Event/chichi/Chichi/figures/Ridgecrest_HAYE_tv_compare.png

── JOHN ──
  Static IoU=0.241  F1=0.388  P=0.898  R=0.247
  Saved: /root/autodl-tmp/Test_on_Event/chichi/Chichi/figures/Ridgecrest_JOHN_iou_curve.png
  Saved: /root/autodl-tmp/Test_on_